# Install packages

In [11]:
!pip install ase scikit-learn scipy numpy -q

In [8]:
!pip install pymatgen -q

# Checking clean slab's geometry

In [12]:
# --- Symmetry check on the clean, relaxed Co slab -----------------------
# Purpose: confirm the slab is symmetric enough that all candidate sites of
# a given type (ontop, bridge, hollow_fcc, hollow_hcp) are physically
# equivalent, so it's valid to run VASP on just one representative site per
# type instead of every candidate found by the Delaunay site-finder.

from ase.io import read
slab = read("/content/CONTCAR_Co")  # relaxed, bare Co surface (no adsorbate)

from pymatgen.io.ase import AseAtomsAdaptor        # ASE <-> pymatgen structure converter
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer  # pymatgen's symmetry detector

# SpacegroupAnalyzer only accepts pymatgen Structure objects, so convert first
pmg_slab = AseAtomsAdaptor.get_structure(slab)

# symprec = tolerance (Angstrom) for how far an atom can sit from its "ideal"
# symmetric position and still be treated as symmetric. Set loosely (0.1 A)
# because this is a DFT-relaxed slab, not a perfect bulk-truncated structure.
sga = SpacegroupAnalyzer(pmg_slab, symprec=0.1)

# Space group symbol: describes the slab's overall symmetry (rotations,
# mirrors, etc). A high-symmetry result (e.g. P3m1) supports reducing
# candidate adsorption sites down to one per type. A low-symmetry result
# (e.g. P1) means the surface isn't uniform and more sites should be kept.
print("Space group:", sga.get_space_group_symbol())

# Number of symmetry operations found (rotations/reflections/translations
# that map the slab onto itself). More operations = more redundancy among
# the 96 candidate sites found earlier.
print("Point group ops:", len(sga.get_symmetry_operations()))

Space group: P3m1
Point group ops: 32


# Cleaning directory from previous run

In [18]:
import shutil
shutil.rmtree("/content/adsorption_sites", ignore_errors=True)
# Run this cell before the main generation cell each time you
# rerun it — it deletes /content/adsorption_sites
#(and everything in it) so you start clean instead of accumulating old
#folders from previous adsorbates

# Placing adsorbate species on surafce

In [19]:
#!/usr/bin/env python3
"""
place_adsorbate_final.py
========================
Place one or more adsorbates on a clean metal slab (cobalt or any HCP/FCC)
at all high-symmetry adsorption sites: on-top, bridge, hollow-hcp, hollow-fcc.

Features
--------
  • Batch processing  – supply as many adsorbate CONTCARs as you like
  • Smart anchor      – auto-detects the binding atom (S > O > N > C > H)
                        with per-adsorbate override support
  • SH2 / anti-parallel fix – robust 3-case rotation so H atoms always
                        point away from the surface even when the CONTCAR
                        stores H below S
  • Safe placement    – guard loop pushes adsorbate up if it overlaps the slab
  • Detailed logging  – prints anchor choice, tail orientation & site counts
  • Colab ready       – zips output and auto-downloads at the end

Output layout
-------------
  /content/adsorption_sites/
      SH2/
          ontop/POSCAR
          bridge/POSCAR
          hollow_hcp/POSCAR
          hollow_fcc/POSCAR
      SH/
          ...
      CO/
          ...

Requirements
------------
  pip install ase scikit-learn scipy
"""

import os
import shutil
import numpy as np
from ase.io import read, write
from ase.constraints import FixAtoms
from sklearn.cluster import KMeans
from scipy.spatial import Delaunay


# ═══════════════════════════════════════════════════════════════════════════════
#  ❶  USER SETTINGS  — edit this block only
# ═══════════════════════════════════════════════════════════════════════════════

# ── Clean slab ────────────────────────────────────────────────────────────────
SLAB_FILE = "/content/CONTCAR_Co"

# ── Adsorbate files  (add / remove as needed) ─────────────────────────────────
ADS_FILES = [
    "/content/CONTCAR_S",
    "/content/CONTCAR_SH",
    "/content/CONTCAR_SH2",
    "/content/CONTCAR_SHCH3",
    "/content/CONTCAR_SCH3",
    # add more paths here …
]

# ── Anchor-atom priority ───────────────────────────────────────────────────────
# The FIRST element in this list that is present in the adsorbate is used as
# the binding atom.  Covers most HDS / surface-chemistry adsorbates.
ANCHOR_PRIORITY = ["S", "O", "N", "C", "H"]

# ── Per-adsorbate overrides (label = filename without "CONTCAR_") ──────────────
# Use this when auto-detection gives the wrong atom.
# Example: CO should bind through C, not O  →  uncomment the line below.
ANCHOR_OVERRIDE = {
    # "CO"  : "C",
    # "CO2" : "C",
}

# ── Slab geometry ─────────────────────────────────────────────────────────────
N_LAYERS       = 5      # total layers in the slab
N_FIXED_LAYERS = 2      # bottom N layers held fixed during VASP relaxation
HEIGHT         = 1.8    # Å above top-layer mean z where the anchor atom sits
NN_CUTOFF      = 3.0    # Å  – max bond for bridge / hollow detection
HCP_THRESH     = 0.75   # Å  – distance threshold for hcp vs fcc hollow
SEED           = 42     # KMeans random seed

# ── Output ────────────────────────────────────────────────────────────────────
OUT_ROOT = "/content/adsorption_sites"


# ═══════════════════════════════════════════════════════════════════════════════
#  ❷  LAYER DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

def get_layer_indices(atoms, n_layers, seed):
    """
    Cluster slab atoms into layers by z-coordinate using KMeans.
    Returns a list of index arrays ordered from top (highest z) to bottom.
    """
    z  = atoms.positions[:, 2].reshape(-1, 1)
    km = KMeans(n_clusters=n_layers, random_state=seed, n_init=10).fit(z)
    means = [atoms.positions[km.labels_ == i, 2].mean() for i in range(n_layers)]
    order = np.argsort(means)[::-1]   # highest z first
    return [np.where(km.labels_ == order[k])[0] for k in range(n_layers)]


# ═══════════════════════════════════════════════════════════════════════════════
#  ❸  SITE DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

def periodic_xy(xy, cell, reps=(-1, 0, 1)):
    """Tile 2-D positions over ±1 periodic images in both a and b directions."""
    a, b = cell[0][:2], cell[1][:2]
    return np.vstack([xy + i * np.array(a) + j * np.array(b)
                      for i in reps for j in reps])


def in_central_cell(xy, cell, tol=1e-3):
    """Return True if the 2-D point lies inside the primary unit cell."""
    A    = np.array([cell[0][:2], cell[1][:2]]).T
    frac = np.linalg.solve(A, xy)
    return np.all(frac >= -tol) and np.all(frac < 1 - tol)


def find_sites(atoms, layers):
    """
    Find all on-top, bridge, hollow-hcp and hollow-fcc sites using the
    Delaunay triangulation of the top-layer atoms (with periodic images).
    Returns a list of dicts: {type, xy, z}.
    """
    top_idx    = layers[0]
    second_idx = layers[1]
    z_surf     = atoms.positions[top_idx, 2].mean()
    top_xy     = atoms.positions[top_idx, :2]
    second_xy  = periodic_xy(atoms.positions[second_idx, :2], atoms.get_cell())

    # ── On-top ──────────────────────────────────────────────────────────────
    sites = [dict(type="ontop", xy=xy) for xy in top_xy]

    # Build tiled Delaunay of top-layer atoms (with periodic images)
    tiled = periodic_xy(top_xy, atoms.get_cell())
    tri   = Delaunay(tiled)

    # ── Bridge: midpoint of each short edge inside the cell ──────────────────
    seen = set()
    for simplex in tri.simplices:
        for i in range(3):
            a_idx = simplex[i]
            b_idx = simplex[(i + 1) % 3]
            key   = tuple(sorted((a_idx, b_idx)))
            if key in seen:
                continue
            seen.add(key)
            p1, p2 = tiled[a_idx], tiled[b_idx]
            if np.linalg.norm(p1 - p2) > NN_CUTOFF:
                continue
            mid = (p1 + p2) / 2
            if in_central_cell(mid, atoms.get_cell()):
                sites.append(dict(type="bridge", xy=mid))

    # ── Hollow: centroid of each short triangle inside the cell ──────────────
    for simplex in tri.simplices:
        pts   = tiled[simplex]
        edges = [np.linalg.norm(pts[i] - pts[(i + 1) % 3]) for i in range(3)]
        if max(edges) > NN_CUTOFF:
            continue
        c = pts.mean(axis=0)
        if not in_central_cell(c, atoms.get_cell()):
            continue
        # hcp hollow: a second-layer atom is directly beneath the centroid
        d = np.linalg.norm(second_xy - c, axis=1)
        t = "hollow_hcp" if d.min() < HCP_THRESH else "hollow_fcc"
        sites.append(dict(type=t, xy=c))

    for s in sites:
        s["z"] = z_surf
    return sites


def dedupe(sites, tol=0.3):
    """Remove duplicate sites of the same type within `tol` Å of each other."""
    kept = []
    for s in sites:
        if not any(
            k["type"] == s["type"] and
            np.linalg.norm(np.array(k["xy"]) - np.array(s["xy"])) < tol
            for k in kept
        ):
            kept.append(s)
    return kept


# ═══════════════════════════════════════════════════════════════════════════════
#  ❹  ANCHOR DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

def detect_anchor(adsorbate, label, priority, override):
    """
    Choose which atom in the adsorbate binds to the surface.

    Priority order
    --------------
    1. ANCHOR_OVERRIDE[label]  — explicit user override
    2. ANCHOR_PRIORITY         — first element found wins (S > O > N > C > H)
    3. Among multiple atoms of the chosen element, pick the one with the
       lowest z in the input file (closest to the surface orientation stored
       in the CONTCAR).

    Returns (anchor_index, anchor_symbol).
    """
    symbols = [a.symbol for a in adsorbate]

    # Step 1: explicit override
    if label in override:
        chosen_sym = override[label]
        if chosen_sym not in symbols:
            raise ValueError(
                f"[{label}] ANCHOR_OVERRIDE specifies '{chosen_sym}' "
                f"but that element is not in the adsorbate ({set(symbols)})."
            )
    else:
        # Step 2: walk priority list
        chosen_sym = None
        for sym in priority:
            if sym in symbols:
                chosen_sym = sym
                break
        if chosen_sym is None:
            raise ValueError(
                f"[{label}] None of the ANCHOR_PRIORITY elements {priority} "
                f"were found in the adsorbate (elements: {set(symbols)}). "
                "Add the correct element to ANCHOR_PRIORITY or ANCHOR_OVERRIDE."
            )

    # Step 3: pick lowest-z candidate if multiple atoms of the same element
    candidates = [i for i, a in enumerate(adsorbate) if a.symbol == chosen_sym]
    anchor_idx = min(candidates, key=lambda i: adsorbate.positions[i, 2])
    return anchor_idx, chosen_sym


# ═══════════════════════════════════════════════════════════════════════════════
#  ❺  ROTATION  (the core fix)
# ═══════════════════════════════════════════════════════════════════════════════

def rotation_matrix_align(tail, target=None):
    """
    Return a 3×3 rotation matrix R such that  R @ tail  ≈  target.

    Three cases are handled explicitly:

    Case 1 — tail ≈ target (already aligned, dot ≈ +1)
        R = identity  (no rotation needed)

    Case 2 — tail ≈ −target (anti-parallel, dot ≈ −1)  ← THE SH2 FIX
        The cross product is ≈ zero so the general Rodrigues formula
        produces a zero axis and silently does nothing.
        Instead we apply a 180° flip around the x-axis.

    Case 3 — general angle
        Standard Rodrigues' rotation formula around the cross-product axis.

    Parameters
    ----------
    tail   : array-like, shape (3,) — vector to rotate FROM (need not be unit)
    target : array-like, shape (3,) — unit vector to rotate TO
             defaults to [0, 0, +1] (point tail upward, away from surface)

    Returns
    -------
    R : ndarray, shape (3, 3)
    """
    if target is None:
        target = np.array([0.0, 0.0, 1.0])

    tail   = np.asarray(tail,   dtype=float)
    target = np.asarray(target, dtype=float)
    tail   = tail   / np.linalg.norm(tail)
    target = target / np.linalg.norm(target)

    axis = np.cross(tail, target)
    an   = np.linalg.norm(axis)
    dot  = float(np.dot(tail, target))

    if an > 1e-6:
        # ── Case 3: general rotation via Rodrigues' formula ──────────────────
        axis  /= an
        angle  = np.arccos(np.clip(dot, -1.0, 1.0))
        K = np.array([[      0, -axis[2],  axis[1]],
                      [ axis[2],       0, -axis[0]],
                      [-axis[1],  axis[0],       0]])
        R = np.eye(3) + np.sin(angle) * K + (1.0 - np.cos(angle)) * (K @ K)

    elif dot < 0:
        # ── Case 2: anti-parallel (tail ≈ [0,0,−1]) — 180° flip ─────────────
        # This is the fix for SH2 (and any molecule stored with its tail
        # pointing straight down in the CONTCAR).
        R = np.diag([1.0, -1.0, -1.0])   # 180° around x-axis

    else:
        # ── Case 1: already aligned ───────────────────────────────────────────
        R = np.eye(3)

    return R


# ═══════════════════════════════════════════════════════════════════════════════
#  ❻  PLACEMENT
# ═══════════════════════════════════════════════════════════════════════════════

def place_adsorbate(slab, ads, anchor_idx, xy, z_surf, height):
    """
    Place a copy of `ads` on `slab` at surface site (xy, z_surf + height).

    Steps
    -----
    1. Compute the tail vector (anchor → centroid of remaining atoms).
    2. Rotate the molecule so the tail points upward (+z), i.e. the anchor
       faces the surface.  Uses rotation_matrix_align which correctly handles
       the anti-parallel case (see Case 2 above).
    3. Translate the anchor to the target site position.
    4. Guard loop: nudge the adsorbate up in 0.1 Å steps until no
       adsorbate–slab distance is shorter than 1.0 Å.

    Parameters
    ----------
    slab       : ASE Atoms — the clean slab
    ads        : ASE Atoms — the isolated adsorbate molecule
    anchor_idx : int       — index of the binding atom within `ads`
    xy         : array (2,)— x, y coordinates of the adsorption site
    z_surf     : float     — mean z of the top layer
    height     : float     — Å above z_surf for the anchor atom

    Returns
    -------
    combined : ASE Atoms — slab + adsorbate
    """
    mol = ads.copy()
    n   = len(mol)

    if n > 1:
        anchor_pos = mol.positions[anchor_idx].copy()
        others     = [i for i in range(n) if i != anchor_idx]

        # tail = direction from anchor to centroid of all other atoms
        tail = mol.positions[others].mean(axis=0) - anchor_pos
        tn   = np.linalg.norm(tail)

        if tn > 1e-6:
            R = rotation_matrix_align(tail / tn)   # rotate tail → +z

            # Rotate around the anchor (shift to origin, rotate, shift back)
            mol.positions = (R @ (mol.positions - anchor_pos).T).T + anchor_pos

    # Translate so the anchor atom lands exactly at the site
    target_pos = np.array([xy[0], xy[1], z_surf + height])
    mol.translate(target_pos - mol.positions[anchor_idx])

    # Combine slab and adsorbate
    combined = slab.copy() + mol
    n_slab   = len(slab)

    # Guard: avoid unphysical short contacts with the slab
    for _ in range(50):
        d = combined.get_all_distances()
        if np.min(d[n_slab:, :n_slab]) >= 1.0:
            break
        combined.positions[n_slab:, 2] += 0.1

    return combined


# ═══════════════════════════════════════════════════════════════════════════════
#  ❼  PER-ADSORBATE WORKFLOW
# ═══════════════════════════════════════════════════════════════════════════════

def process_adsorbate(slab, layers, fixed, ads_path):
    """Full pipeline for one adsorbate file: detect anchor, find sites, write POSCARs."""

    # Derive label from filename: CONTCAR_SH2 → "SH2"
    label = (os.path.basename(ads_path)
             .replace("CONTCAR_", "")
             .replace("POSCAR_", ""))

    print(f"\n{'═' * 62}")
    print(f"  Adsorbate : {label}   ({ads_path})")
    print(f"{'═' * 62}")

    adsorbate  = read(ads_path)
    anchor_idx, anchor_sym = detect_anchor(
        adsorbate, label, ANCHOR_PRIORITY, ANCHOR_OVERRIDE
    )

    # ── Diagnostic output ────────────────────────────────────────────────────
    print(f"  Anchor atom      : {anchor_sym}  (index {anchor_idx} in adsorbate)")
    print(f"  Anchor position  : {np.round(adsorbate.positions[anchor_idx], 3)}")

    if len(adsorbate) > 1:
        others = [i for i in range(len(adsorbate)) if i != anchor_idx]
        tail   = adsorbate.positions[others].mean(axis=0) - adsorbate.positions[anchor_idx]
        tail_n = tail / np.linalg.norm(tail)
        dot    = float(np.dot(tail_n, [0, 0, 1]))

        if dot < -0.95:
            diag = "anti-parallel  →  180° flip applied  ✓  (SH2-type fix)"
        elif dot > 0.95:
            diag = "already upward  →  no rotation needed"
        else:
            diag = f"general  ({np.degrees(np.arccos(np.clip(dot, -1, 1))):.1f}°)  →  Rodrigues rotation"

        print(f"  Tail vector      : {np.round(tail_n, 3)}")
        print(f"  Orientation      : {diag}")
    else:
        print("  Single-atom adsorbate — no rotation required.")

    # ── Site detection ────────────────────────────────────────────────────────
    sites   = dedupe(find_sites(slab, layers))
    by_type = {}
    for s in sites:
        by_type.setdefault(s["type"], []).append(s)

    print()
    for site_type, site_list in by_type.items():
        print(f"  {site_type:14s}: {len(site_list)} candidate(s)")

    # One representative per site type (valid for high-symmetry P3m1 slab)
    chosen_sites = {t: s_list[0] for t, s_list in by_type.items()}

    # ── Write POSCARs ─────────────────────────────────────────────────────────
    print()
    for site_type, site in chosen_sites.items():
        system = place_adsorbate(
            slab, adsorbate, anchor_idx,
            site["xy"], site["z"], HEIGHT
        )
        system.set_constraint(FixAtoms(indices=fixed))

        folder = os.path.join(OUT_ROOT, label, site_type)
        os.makedirs(folder, exist_ok=True)
        out_path = os.path.join(folder, "POSCAR")
        write(out_path, system, format="vasp", vasp5=True, direct=True)
        print(f"  wrote  →  {out_path}")


# ═══════════════════════════════════════════════════════════════════════════════
#  ❽  MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    os.makedirs(OUT_ROOT, exist_ok=True)

    # ── Load slab ─────────────────────────────────────────────────────────────
    print(f"\nSlab file  : {SLAB_FILE}")
    slab   = read(SLAB_FILE)
    layers = get_layer_indices(slab, N_LAYERS, SEED)
    fixed  = np.concatenate(layers[-N_FIXED_LAYERS:]).tolist()

    print(f"Slab atoms : {len(slab)}")
    print(f"Layers     : {N_LAYERS}  (top → bottom z means: "
          + "  ".join(f"{slab.positions[lyr, 2].mean():.2f}" for lyr in layers) + " Å)")
    print(f"Fixed idx  : bottom {N_FIXED_LAYERS} layer(s)  ({len(fixed)} atoms)")

    # ── Check for missing files ───────────────────────────────────────────────
    missing = [f for f in ADS_FILES if not os.path.isfile(f)]
    if missing:
        print(f"\n⚠  The following files were NOT found and will be skipped:")
        for m in missing:
            print(f"     {m}")

    # ── Process each adsorbate ────────────────────────────────────────────────
    ok, fail = 0, 0
    for ads_path in ADS_FILES:
        if not os.path.isfile(ads_path):
            continue
        try:
            process_adsorbate(slab, layers, fixed, ads_path)
            ok += 1
        except Exception as exc:
            print(f"\n  ✗  ERROR processing {ads_path}")
            print(f"     {exc}")
            fail += 1

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'═' * 62}")
    print(f"  Finished.  {ok} adsorbate(s) OK  |  {fail} error(s)")
    print(f"  Output root : {OUT_ROOT}")
    print(f"{'═' * 62}\n")

    # ── Zip and download (Google Colab) ───────────────────────────────────────
    zip_path = "/content/adsorption_sites"
    shutil.make_archive(zip_path, "zip", OUT_ROOT)
    print(f"Archive created : {zip_path}.zip")

    try:
        from google.colab import files
        files.download(f"{zip_path}.zip")
        print("Download triggered.")
    except ImportError:
        print("(Not running in Colab — skipping auto-download.)")


if __name__ == "__main__":
    main()


Slab file  : /content/CONTCAR_Co
Slab atoms : 80
Layers     : 5  (top → bottom z means: 9.93  8.01  6.03  4.06  2.03 Å)
Fixed idx  : bottom 2 layer(s)  (32 atoms)

══════════════════════════════════════════════════════════════
  Adsorbate : S   (/content/CONTCAR_S)
══════════════════════════════════════════════════════════════
  Anchor atom      : S  (index 0 in adsorbate)
  Anchor position  : [7.5  7.5  7.96]
  Single-atom adsorbate — no rotation required.

  ontop         : 16 candidate(s)
  bridge        : 48 candidate(s)
  hollow_fcc    : 16 candidate(s)
  hollow_hcp    : 16 candidate(s)

  wrote  →  /content/adsorption_sites/S/ontop/POSCAR
  wrote  →  /content/adsorption_sites/S/bridge/POSCAR
  wrote  →  /content/adsorption_sites/S/hollow_fcc/POSCAR
  wrote  →  /content/adsorption_sites/S/hollow_hcp/POSCAR

══════════════════════════════════════════════════════════════
  Adsorbate : SH   (/content/CONTCAR_SH)
══════════════════════════════════════════════════════════════
  Anchor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered.


# Checking final adsorption site after DFT

In [ ]:
import glob
import re
import numpy as np
from ase.io import read
from ase.neighborlist import NeighborList, natural_cutoffs

METAL_SYMBOL = "Co"
ANCHOR_SYMBOL = "S"          # same as your placement anchor
CONTCAR_PATTERN = "/content/CONTCAR_Co_*"   # matches CONTCAR_Co_H2S_ontop, CONTCAR_Co_H2S_bridge, ...
POSCAR_ROOT = "/content/adsorption_sites"   # your pre-VASP POSCARs, for dissociation comparison
COORD_CUTOFF = 3.2           # distance (A) under which a metal atom counts as "bonded" to the anchor

def check_dissociation(initial_ads, final_ads):
    try:
        inl = NeighborList(natural_cutoffs(initial_ads), self_interaction=False, bothways=True)
        inl.update(initial_ads)
        fnl = NeighborList(natural_cutoffs(final_ads), self_interaction=False, bothways=True)
        fnl.update(final_ads)
        for i in range(len(initial_ads)):
            if set(inl.get_neighbors(i)[0]) != set(fnl.get_neighbors(i)[0]):
                return True
        return False
    except Exception:
        return True

def check_desorption(system, n_slab_atoms, cushion=1.5):
    try:
        cutoffs = [c * cushion for c in natural_cutoffs(system)]
        nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
        nl.update(system)
        for ads_idx in range(n_slab_atoms, len(system)):
            if any(n < n_slab_atoms for n in nl.get_neighbors(ads_idx)[0]):
                return False
        return True
    except Exception:
        return True

def parse_label_site(path):
    # e.g. CONTCAR_Co_H2S_ontop -> label="H2S", site="ontop"
    fname = path.split("/")[-1]
    m = re.match(r"CONTCAR_Co_(.+)_(ontop|bridge|hollow_fcc|hollow_hcp)$", fname)
    if not m:
        raise ValueError(f"Filename doesn't match expected pattern: {fname}")
    return m.group(1), m.group(2)

def analyze_final_structure(contcar_path):
    ads_label, site_label = parse_label_site(contcar_path)
    final_system = read(contcar_path)

    # matching pre-VASP POSCAR, for the dissociation check and slab atom count
    poscar_path = f"{POSCAR_ROOT}/{ads_label}_{site_label}/POSCAR"
    initial_system = read(poscar_path)
    n_slab = sum(1 for a in initial_system if a.symbol == METAL_SYMBOL)

    initial_ads = initial_system[n_slab:].copy()
    final_ads = final_system[n_slab:].copy()

    dissociated = check_dissociation(initial_ads, final_ads) if len(initial_ads) > 1 else False
    desorbed = check_desorption(final_system, n_slab)

    anchor_idx = [i for i, a in enumerate(final_system) if a.symbol == ANCHOR_SYMBOL][0]
    metal_idx = [i for i, a in enumerate(final_system) if a.symbol == METAL_SYMBOL]

    top_metal_z = final_system.positions[metal_idx, 2].max()
    height = final_system.positions[anchor_idx, 2] - top_metal_z
    dists = sorted(final_system.get_distances(anchor_idx, metal_idx, mic=True))
    n_coord = sum(d < COORD_CUTOFF for d in dists)

    if n_coord == 1:
        final_site_type = "ontop"
    elif n_coord == 2:
        final_site_type = "bridge"
    elif n_coord >= 3:
        final_site_type = "hollow (fcc/hcp not distinguished here)"
    else:
        final_site_type = "desorbed / undercoordinated"

    return {
        "file": contcar_path.split("/")[-1],
        "intended_site": site_label,
        "final_site_by_coordination": final_site_type,
        "height_above_surface": round(height, 3),
        "anchor_metal_dist": round(dists[0], 3),
        "coordination_number": n_coord,
        "dissociated": dissociated,
        "desorbed": desorbed,
        "site_moved": site_label not in final_site_type,
    }

results = []
for path in sorted(glob.glob(CONTCAR_PATTERN)):
    try:
        results.append(analyze_final_structure(path))
    except Exception as e:
        print(f"Skipping {path}: {e}")

for r in results:
    print(r)